In [ ]:
import os
import json
from datasets import load_dataset
from tqdm import tqdm

def process_dataset_robust(output_dir="./multimodal_data", max_samples=200):
    os.makedirs(output_dir, exist_ok=True)
    images_dir = os.path.join(output_dir, "images")
    os.makedirs(images_dir, exist_ok=True)
    output_jsonl_path = os.path.join(output_dir, "raw_input_multimodal.jsonl")

    print("1. 正在加载数据集...")
    dataset = load_dataset("hiyouga/geometry3k", split="train")
    print(f"成功加载，总共有 {len(dataset)} 条原始题。")

    formatted_data = []
    
    for idx in tqdm(range(min(len(dataset), max_samples))):
        row = dataset[idx]
        
        # 1. 强力提取图像
        image = None
        for key in ["image", "img", "decoded_image", "diagram"]:
            if key in row and row[key] is not None:
                image = row[key]
                break
                
        # 2. 强力提取文本（如果字段缺失就找可能包含题目的字段）
        question = ""
        for key in ["problem_text", "question", "text", "query", "problem"]:
            if key in row and row[key]:
                question = str(row[key])
                break
        if not question:
            # 如果没有单独立题，把所有字符串字段拼起来作为题目
            question = f"Geometry problem with diagram (ID: {idx})"

        # 3. 强力提取解答/选项
        solution_content = ""
        for key in ["solution", "rational", "explanation", "choices", "answer"]:
            if key in row and row[key]:
                solution_content = str(row[key])
                break
        if not solution_content:
            solution_content = "Find the missing geometric value based on the figure."

        # 4. 保存图像
        image_save_path = os.path.join(images_dir, f"geo_{idx}.png")
        if image is not None:
            try:
                # 兼容 PIL Image 或字节流
                if hasattr(image, "save"):
                    image.save(image_save_path)
                else:
                    with open(image_save_path, "wb") as img_f:
                        img_f.write(image)
            except Exception as e:
                image_save_path = None
        else:
            image_save_path = None

        # 5. 构造多模态评估条目（只要有题就必须生成，不跳过！）[cite: 1]
        sample = {
            "id": f"mm_geo_{idx}",
            "image_path": os.path.abspath(image_save_path) if image_save_path else None,
            "question": question,
            "previous_steps": "None",
            "now_step": f"Step 1: {solution_content}",
            "human_label": "Yes"  # 基准样本默认设为 Yes[cite: 1]
        }
        formatted_data.append(sample)

    # 6. 保存为 JSONL 文件
    with open(output_jsonl_path, "w", encoding="utf-8") as f:
        for item in formatted_data:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")

    print(f"\n🎉 提取成功！这次成功生成了 {len(formatted_data)} 条样本！")
    print(f"数据已保存在: {os.path.abspath(output_jsonl_path)}")

if __name__ == "__main__":
    process_dataset_robust(max_samples=200)

In [ ]:
import os
import json
import base64
import re
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

# ================= 1. 配置参数 =================
API_KEY = ""
URL = "https://api.siliconflow.cn/v1/chat/completions"
MODEL_NAME = "Qwen/Qwen3-VL-8B-Instruct"

MAX_WORKERS = 10  
INPUT_FILE = "./multimodal_data/raw_input_multimodal.jsonl"
OUTPUT_FILE = "./multimodal_data/filtered_mm_seed_data.jsonl"

# ================= 2. 多模态 Prompt =================
MM_PROMPT = """You are an expert multimodal mathematics teacher. Your task is to verify the correctness of the "Now Step" based on the provided Image, Question, and context.

Analysis Dimensions:
1. **Visual & Geometric Analysis**: Check if conditions derived from the image are accurate.
2. **Now Step Analysis**: Explain what the current step aims to solve.
3. **Data Source Analysis**: Verify values/variables coming from the problem or image.
4. **Calculation Analysis**: Check if calculations are correct.

Conclusion:
Conclude your evaluation. At the very end, strictly output your verdict in the following format:
Verification: Is the step correct (Yes/No)? X
(where X is either Yes or No).

Question: {question}
Previous Steps: {previous_steps}
Now Step: {now_step}
Reply:"""

def encode_image(image_path):
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

def parse_model_verdict(text):
    """
    鲁棒解析模型输出中的 Yes / No（兼容 Markdown 粗体、标点和换行）
    """
    # 移除常见的 Markdown 粗体符号，避免干扰正则
    clean_text = text.replace("**", "").replace("__", "")

    # 1. 优先匹配标准格式: Verification: Is the step correct (Yes/No)? Yes/No
    match = re.search(r"Verification:\s*Is the step correct\s*\(Yes/No\)\?\s*(Yes|No)", clean_text, re.IGNORECASE)
    if match:
        return match.group(1).capitalize()

    # 2. 宽松匹配 Verification 开头的行
    match = re.search(r"Verification:.*?(Yes|No)", clean_text, re.IGNORECASE)
    if match:
        return match.group(1).capitalize()

    # 3. 兜底匹配：提取文本最后 50 个字符中的 Yes 或 No
    tail_text = clean_text[-50:]
    tail_match = re.findall(r"\b(Yes|No)\b", tail_text, re.IGNORECASE)
    if tail_match:
        return tail_match[-1].capitalize()

    return None

# ================= 3. 单条数据请求与验证 =================
def process_item(item):
    image_path = item.get("image_path")
    if not image_path or not os.path.exists(image_path):
        return None, "图片缺失"

    try:
        base64_img = encode_image(image_path)
    except Exception as e:
        return None, f"图片读取失败: {e}"

    prompt_text = MM_PROMPT.format(
        question=item["question"],
        previous_steps=item.get("previous_steps", "None"),
        now_step=item["now_step"]
    )

    payload = {
        "model": MODEL_NAME,
        "messages": [
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt_text},
                    {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{base64_img}"}}
                ]
            }
        ],
        "temperature": 0.1,
        "max_tokens": 1024
    }
    headers = {"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"}

    try:
        res = requests.post(URL, headers=headers, json=payload, timeout=60)
        if res.status_code != 200:
            return None, f"API 状态码异常: {res.status_code}"

        cot_text = res.json()["choices"][0]["message"]["content"]
        pred_label = parse_model_verdict(cot_text)

        if not pred_label:
            return None, "正则未捕获到结论"

        ground_truth = item.get("human_label", "Yes")
        # 硬过滤：预测结论与真实标签一致时保留
        if pred_label == ground_truth:
            return {
                "id": item.get("id"),
                "image_path": item["image_path"],
                "question": item["question"],
                "previous_steps": item.get("previous_steps", "None"),
                "now_step": item["now_step"],
                "human_label": ground_truth,
                "generated_cot": cot_text
            }, f"成功 (判定为 {pred_label})"
        else:
            return None, f"标签不符丢弃 (预测:{pred_label}, 实际:{ground_truth})"

    except Exception as e:
        return None, f"网络请求异常: {e}"

# ================= 4. 主控运行 =================
def main():
    if not os.path.exists(INPUT_FILE):
        print(f"找不到输入文件: {INPUT_FILE}")
        return

    with open(INPUT_FILE, "r", encoding="utf-8") as f:
        items = [json.loads(line) for line in f]

    print(f"🚀 读取到 {len(items)} 条数据，启动 Qwen3-VL-8B 多线程批量生成...")

    success_count = 0
    discard_count = 0
    parse_err_count = 0

    with open(OUTPUT_FILE, "a", encoding="utf-8") as out_f:
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            future_to_item = {executor.submit(process_item, item): item for item in items}
            pbar = tqdm(as_completed(future_to_item), total=len(items))

            for future in pbar:
                result, msg = future.result()
                if result:
                    out_f.write(json.dumps(result, ensure_ascii=False) + "\n")
                    out_f.flush()
                    success_count += 1
                elif "丢弃" in msg:
                    discard_count += 1
                elif "正则" in msg:
                    parse_err_count += 1

                pbar.set_description(f"成功: {success_count} | 过滤丢弃: {discard_count} | 解析失败: {parse_err_count}")

    print(f"\n✅ 全部完成！")
    print(f"📊 最终统计: 成功写入 {success_count} 条，过滤丢弃 {discard_count} 条，解析失败 {parse_err_count} 条。")
    print(f"📁 结果已落盘至: {os.path.abspath(OUTPUT_FILE)}")


In [ ]:
# ==============================================================================
Scale-up Pipeline
# ==============================================================================

import os
import sys
import json
import base64
import random
import re
import time
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# ================= 1. 核心参数配置 =================
API_KEY = ""  
URL = "https://api.siliconflow.cn/v1/chat/completions"
MODEL_NAME = "Qwen/Qwen3-VL-8B-Instruct"  

MAX_WORKERS = 12          
TARGET_RAW_SAMPLES = 1500 
MAX_RETRIES = 4          

DATA_DIR = "./multimodal_data"
IMAGES_DIR = os.path.join(DATA_DIR, "images")
RAW_JSONL_PATH = os.path.join(DATA_DIR, "raw_scale_input.jsonl")
OUTPUT_JSONL_PATH = os.path.join(DATA_DIR, "filtered_mm_seed_data.jsonl")

os.makedirs(IMAGES_DIR, exist_ok=True)

# ================= 2. 多模态 CoT 验证 Prompt =================
MM_PROMPT_TEMPLATE = """You are an expert multimodal mathematics teacher. Your task is to verify the correctness of the "Now Step" based on the provided Image, Question, and context.

Analysis Dimensions:
1. **Visual & Geometric Analysis**: Check if conditions derived from the image are accurate (e.g., lines, points, angles, lengths).
2. **Now Step Analysis**: Explain what the current step aims to solve and check its mathematical soundness.
3. **Data Source Analysis**: Verify whether values or formulas come legitimately from the problem or image.
4. **Calculation Analysis**: Check if arithmetic or algebraic derivations are strictly correct.

Conclusion:
Conclude your evaluation. At the very end, strictly output your verdict in the following format:
Verification: Is the step correct (Yes/No)? X
(where X is either Yes or No).

Question: {question}
Previous Steps: {previous_steps}
Now Step: {now_step}
Reply:"""


# ================= 3. 步骤一：数据提取与正负样本构造 =================
def perturb_answer(answer_str):
    """通过数值扰动或替换制造合理的负例 (Negative Step)"""
    answer_str = str(answer_str).strip()
    try:
        val = float(answer_str)
        if val.is_integer():
            val = int(val)
        delta = random.choice([-3, -2, -1, 1, 2, 3, 5])
        perturbed = val + delta
        if perturbed <= 0 and val > 0:
            perturbed = val + 2
        return str(perturbed)
    except ValueError:
        choices = ["A", "B", "C", "D"]
        if answer_str.upper() in choices:
            other_choices = [c for c in choices if c != answer_str.upper()]
            return random.choice(other_choices)
        return answer_str + " + 1"


def prepare_raw_data():
    if os.path.exists(RAW_JSONL_PATH):
        with open(RAW_JSONL_PATH, "r", encoding="utf-8") as f:
            lines = f.readlines()
        if len(lines) >= TARGET_RAW_SAMPLES * 2:
            print(f"[*] 发现已有原始提取数据 {len(lines)} 条，跳过重新下载。")
            return

    from datasets import load_dataset
    print(f"[*] 正在从 Hugging Face 加载 geometry3k 数据集...")
    dataset = load_dataset("hiyouga/geometry3k", split="train")
    total_avail = len(dataset)
    num_to_fetch = min(total_avail, TARGET_RAW_SAMPLES)
    print(f"[*] 数据集加载完毕，共 {total_avail} 条，计划提取前 {num_to_fetch} 条...")

    records = []
    for idx in tqdm(range(num_to_fetch), desc="提取题目并保存图片"):
        row = dataset[idx]
        problem_text = row.get("problem", "").replace("<image>", "").strip()
        answer_text = str(row.get("answer", "")).strip()

        img_list = row.get("images", [])
        if not img_list or img_list[0] is None:
            continue

        pil_image = img_list[0]
        img_filename = f"scale_geo_{idx}.png"
        img_save_path = os.path.join(IMAGES_DIR, img_filename)

        if not os.path.exists(img_save_path):
            if pil_image.mode == "RGBA":
                pil_image = pil_image.convert("RGB")
            pil_image.save(img_save_path)

        # 构造正例 (Positive Step, Label: Yes)
        pos_sample = {
            "id": f"geo_{idx}_pos",
            "image_path": os.path.abspath(img_save_path),
            "question": problem_text if problem_text else "Find the solution based on the geometric figure.",
            "previous_steps": "None",
            "now_step": f"Step 1: Based on the geometric theorems and values shown in the figure, we derive that the answer is {answer_text}.",
            "human_label": "Yes"
        }
        records.append(pos_sample)

        # 构造负例扰动 (Negative Step, Label: No)
        wrong_ans = perturb_answer(answer_text)
        neg_sample = {
            "id": f"geo_{idx}_neg",
            "image_path": os.path.abspath(img_save_path),
            "question": problem_text if problem_text else "Find the solution based on the geometric figure.",
            "previous_steps": "None",
            "now_step": f"Step 1: Applying the geometric relations in the diagram incorrectly leads to {wrong_ans}.",
            "human_label": "No"
        }
        records.append(neg_sample)

    with open(RAW_JSONL_PATH, "w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print(f"[+] 原始待推理样本构造完成！共写入 {len(records)} 条正负样本至: {RAW_JSONL_PATH}")


# ================= 4. 辅助函数与模型解析 =================
def encode_image(image_path):
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")


def parse_model_verdict(text):
    """多级鲁棒正则提取 Yes / No"""
    clean_text = text.replace("**", "").replace("__", "")
    match = re.search(r"Verification:\s*Is the step correct\s*\(Yes/No\)\?\s*(Yes|No)", clean_text, re.IGNORECASE)
    if match:
        return match.group(1).capitalize()
    match = re.search(r"Verification:.*?(Yes|No)", clean_text, re.IGNORECASE)
    if match:
        return match.group(1).capitalize()
    tail_text = clean_text[-60:]
    tail_match = re.findall(r"\b(Yes|No)\b", tail_text, re.IGNORECASE)
    if tail_match:
        return tail_match[-1].capitalize()
    return None


# ================= 5. 单条请求与自动重试逻辑 =================
def process_single_item(item):
    image_path = item.get("image_path")
    if not image_path or not os.path.exists(image_path):
        return None, "图片丢失"

    try:
        base64_img = encode_image(image_path)
    except Exception as e:
        return None, f"图片编码异常: {e}"

    prompt_text = MM_PROMPT_TEMPLATE.format(
        question=item["question"],
        previous_steps=item.get("previous_steps", "None"),
        now_step=item["now_step"]
    )

    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json"
    }

    payload = {
        "model": MODEL_NAME,
        "messages": [
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt_text},
                    {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{base64_img}"}}
                ]
            }
        ],
        "temperature": 0.1,
        "max_tokens": 1024
    }

    # 指数退避重试循环
    for attempt in range(MAX_RETRIES):
        try:
            res = requests.post(URL, headers=headers, json=payload, timeout=75)
            if res.status_code == 200:
                data = res.json()
                cot_text = data["choices"][0]["message"]["content"]
                pred_label = parse_model_verdict(cot_text)

                if not pred_label:
                    return None, "正则未匹配"

                ground_truth = item.get("human_label", "Yes")
                # 硬过滤：预测一致才写入
                if pred_label == ground_truth:
                    record = {
                        "id": item.get("id"),
                        "image_path": item["image_path"],
                        "question": item["question"],
                        "previous_steps": item.get("previous_steps", "None"),
                        "now_step": item["now_step"],
                        "human_label": ground_truth,
                        "generated_cot": cot_text
                    }
                    return record, f"通过({pred_label})"
                else:
                    return None, f"过滤丢弃(预:{pred_label}/真:{ground_truth})"

            elif res.status_code == 429:
                sleep_sec = (2 ** attempt) + random.uniform(1.0, 3.0)
                time.sleep(sleep_sec)
                continue
            else:
                return None, f"HTTP_{res.status_code}"

        except requests.exceptions.RequestException:
            sleep_sec = (2 ** attempt) + 1.0
            time.sleep(sleep_sec)
            continue
        except Exception as e:
            return None, f"未知错误: {e}"

    return None, "重试耗尽"


# ================= 6. 主流水线入口 =================
def run_pipeline():
    prepare_raw_data()

    with open(RAW_JSONL_PATH, "r", encoding="utf-8") as f:
        all_raw_items = [json.loads(line) for line in f]

    done_ids = set()
    if os.path.exists(OUTPUT_JSONL_PATH):
        with open(OUTPUT_JSONL_PATH, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    done_ids.add(json.loads(line).get("id"))
                except Exception:
                    pass

    pending_items = [item for item in all_raw_items if item.get("id") not in done_ids]
    print(f"[*] 原始数据总量: {len(all_raw_items)} | 已成功落盘: {len(done_ids)} | 本次待跑: {len(pending_items)}")

    if not pending_items:
        print("[+] 所有数据已全部处理完毕！无需重复运行。")
        return

    success_cnt = len(done_ids)
    discard_cnt = 0
    err_cnt = 0

    with open(OUTPUT_JSONL_PATH, "a", encoding="utf-8") as out_f:
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            future_to_item = {executor.submit(process_single_item, item): item for item in pending_items}
            pbar = tqdm(as_completed(future_to_item), total=len(pending_items), desc="通宵批跑中")

            for future in pbar:
                result, status_msg = future.result()
                if result:
                    out_f.write(json.dumps(result, ensure_ascii=False) + "\n")
                    out_f.flush()
                    success_cnt += 1
                elif "过滤丢弃" in status_msg:
                    discard_cnt += 1
                else:
                    err_cnt += 1

                pbar.set_description(f"累计成功: {success_cnt} | 丢弃: {discard_cnt} | 失败: {err_cnt}")

    print("\n" + "="*50)
    print(f"[+] 批跑结束！")
    print(f"[+] 最终高质量 Seed 数据集: {OUTPUT_JSONL_PATH}")
    print(f"[+] 累计可用样本数: {success_cnt} 条")
    print("="*50)

if __name__ == "__main__":
    run_pipeline()